In [1]:
# =============================================================================
# MÓDULO III – SUBMÓDULO 1: CONSTRUYE MODELOS CON IA
# PRÁCTICA 1: Machine Learning clásico con scikit-learn
# Dataset: Iris  |  Modelo: Random Forest  |  21 de mayo de 2026
# =============================================================================
# Instrucciones:
#   1. Ejecuta celda por celda en Jupyter, o bien:
#      python practica_1_iris_randomforest.py
#   2. Observa cada salida antes de continuar al siguiente bloque.
#   3. Modifica los hiperparámetros marcados con ★ y compara resultados.
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')           # evita errores de GUI en entornos sin pantalla
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# ─────────────────────────────────────────────────────────────────────────────
# BLOQUE 1 ── Cargar y explorar el dataset
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("  BLOQUE 1 – EXPLORACIÓN DEL DATASET")
print("=" * 60)

iris = load_iris()

# Convertir a DataFrame para exploración cómoda
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["especie"] = iris.target_names[iris.target]

print(f"\nForma del dataset:  {df.shape}   ({df.shape[0]} muestras, {df.shape[1]} columnas)")
print(f"Características:    {iris.feature_names}")
print(f"Clases:             {list(iris.target_names)}")

print("\n── Primeras 5 filas ─────────────────────────────────")
print(df.head())

print("\n── Estadísticas descriptivas ────────────────────────")
print(df.describe().round(2))

print("\n── Distribución de clases ───────────────────────────")
print(df["especie"].value_counts())
# Esperado: 50 muestras por clase → dataset BALANCEADO

# ─────────────────────────────────────────────────────────────────────────────
# BLOQUE 2 ── Preprocesamiento
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  BLOQUE 2 – PREPROCESAMIENTO")
print("=" * 60)

X = iris.data    # matriz de características (150×4)
y = iris.target  # vector de etiquetas  (150,)

# ★ PARÁMETRO: cambia test_size (0.1 – 0.4) y observa cómo cambia el accuracy
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,     # ★ 20% para prueba
    random_state=42,    # semilla para reproducibilidad
    stratify=y,         # mantiene la proporción de clases en cada split
)

print(f"\nTamaño train:  {X_train.shape[0]} muestras  ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Tamaño test:   {X_test.shape[0]} muestras  ({X_test.shape[0]/len(X)*100:.0f}%)")

# Normalización
# IMPORTANTE: fit_transform solo en TRAIN; transform en TEST (evitar data leakage)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"\nMedia train (antes escalar):  {X_train.mean(axis=0).round(2)}")
print(f"Media train (después escalar): {X_train_sc.mean(axis=0).round(3)}")
print("→ Media ≈ 0 confirma que el scaler funcionó correctamente.")

# ─────────────────────────────────────────────────────────────────────────────
# BLOQUE 3 ── Entrenamiento con Random Forest
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  BLOQUE 3 – ENTRENAMIENTO: RANDOM FOREST")
print("=" * 60)

# ★ PARÁMETROS: experimenta cambiando n_estimators (10, 50, 100, 200)
#               y max_depth (None, 3, 5, 10)
modelo_rf = RandomForestClassifier(
    n_estimators=100,   # ★ número de árboles en el bosque
    max_depth=None,     # ★ profundidad máxima (None = sin límite)
    min_samples_split=2,
    random_state=42,
    n_jobs=-1,          # usa todos los núcleos disponibles
)

modelo_rf.fit(X_train_sc, y_train)
print("\n✅  Modelo entrenado correctamente.")
print(f"   Árboles en el bosque: {modelo_rf.n_estimators}")

# ─────────────────────────────────────────────────────────────────────────────
# BLOQUE 4 ── Evaluación
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  BLOQUE 4 – EVALUACIÓN DEL MODELO")
print("=" * 60)

y_pred = modelo_rf.predict(X_test_sc)
acc    = accuracy_score(y_test, y_pred)

print(f"\n✅  Accuracy en test:  {acc:.4f}  ({acc*100:.2f}%)")
print("\n── Reporte completo ─────────────────────────────────")
print(classification_report(
    y_test, y_pred,
    target_names=iris.target_names,
    digits=4,
))
# Columnas del reporte:
#   precision  = TP / (TP + FP)  → de lo que predigo como X, ¿cuánto es realmente X?
#   recall     = TP / (TP + FN)  → de todos los X reales, ¿cuántos detecté?
#   f1-score   = media armónica de precision y recall
#   support    = cuántas muestras de esa clase hay en test

# Validación cruzada (más robusta que un solo split)
cv_scores = cross_val_score(modelo_rf, X, y, cv=5, scoring="accuracy")
print(f"── Validación cruzada 5-fold ──────────────────────")
print(f"   Scores: {cv_scores.round(4)}")
print(f"   Media ± Desv: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# BLOQUE 5 ── Visualización: Matriz de confusión
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  BLOQUE 5 – MATRIZ DE CONFUSIÓN")
print("=" * 60)

cm = confusion_matrix(y_test, y_pred)
print("\nMatriz (filas=real, columnas=predicho):")
print(cm)
print("\nInterpretación: diagonal principal = predicciones CORRECTAS")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Práctica 1 – Iris con Random Forest", fontsize=14, fontweight="bold")

# Mapa de calor
disp = ConfusionMatrixDisplay(cm, display_labels=iris.target_names)
disp.plot(cmap="Blues", ax=axes[0])
axes[0].set_title("Matriz de Confusión")

# Importancia de características
importancias = modelo_rf.feature_importances_
colores = ["#00D68F" if i == importancias.argmax() else "#1E3A5F" for i in range(4)]
axes[1].barh(iris.feature_names, importancias, color=colores)
axes[1].set_title("Importancia de características")
axes[1].set_xlabel("Importancia relativa")
for i, v in enumerate(importancias):
    axes[1].text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("practica1_resultados.png", dpi=150, bbox_inches="tight")
print("\n📊  Gráfica guardada en: practica1_resultados.png")

print("\n── Importancia de características ──────────────────")
for nombre, imp in sorted(
    zip(iris.feature_names, importancias), key=lambda x: x[1], reverse=True
):
    barra = "█" * int(imp * 40)
    print(f"  {nombre:30s}  {barra:<40s}  {imp:.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# BLOQUE 6 ── Comparativa: Decision Tree vs Random Forest
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  BLOQUE 6 – COMPARATIVA DE MODELOS")
print("=" * 60)

modelo_dt = DecisionTreeClassifier(random_state=42)
modelo_dt.fit(X_train_sc, y_train)
acc_dt = accuracy_score(y_test, modelo_dt.predict(X_test_sc))
cv_dt  = cross_val_score(modelo_dt, X, y, cv=5).mean()
cv_rf  = cross_val_score(modelo_rf, X, y, cv=5).mean()

print(f"\n{'Modelo':<25} {'Test Acc':>10} {'CV 5-fold':>12}")
print("-" * 50)
print(f"{'Decision Tree':<25} {acc_dt:>10.4f} {cv_dt:>12.4f}")
print(f"{'Random Forest':<25} {acc:>10.4f} {cv_rf:>12.4f}")
print("\n→ Random Forest generalmente supera a un árbol solo gracias al ensemble.")

# ─────────────────────────────────────────────────────────────────────────────
# BLOQUE 7 ── Predicción con nuevos datos
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  BLOQUE 7 – PREDICCIÓN CON DATOS NUEVOS")
print("=" * 60)

# Características: [sépalo largo, sépalo ancho, pétalo largo, pétalo ancho]
nuevas_flores = np.array([
    [5.1, 3.5, 1.4, 0.2],   # muy probablemente setosa
    [6.7, 3.0, 5.2, 2.3],   # muy probablemente virginica
    [5.5, 2.6, 4.4, 1.2],   # probablemente versicolor
])

nuevas_escaladas = scaler.transform(nuevas_flores)
predicciones = modelo_rf.predict(nuevas_escaladas)
probabilidades = modelo_rf.predict_proba(nuevas_escaladas)

print(f"\n{'Medidas (s_l, s_w, p_l, p_w)':<35} {'Predicción':<15} {'Confianza'}")
print("-" * 70)
for flor, pred, proba in zip(nuevas_flores, predicciones, probabilidades):
    nombre = iris.target_names[pred]
    confianza = proba.max()
    print(f"  {str(flor):<33} {nombre:<15} {confianza:.2%}")

print("\n✅  Práctica 1 completada.")
print("   Archivos generados:")
print("   └── practica1_resultados.png")
print("\n   Próximo paso: practica_2_mnist_red_neuronal.py")


  BLOQUE 1 – EXPLORACIÓN DEL DATASET

Forma del dataset:  (150, 5)   (150 muestras, 5 columnas)
Características:    ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Clases:             [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]

── Primeras 5 filas ─────────────────────────────────
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

  especie  
0  setosa  
1  setosa  
2  setosa  
3  setosa  
4  setosa  

── Estadísticas descriptivas ────────────────────────
       sepal length (cm)  sepal width (cm)  petal length (c